# 証券営業インテリジェンス ハンズオン
## Part 4: Cortex Search（ドキュメント解析 + セマンティック検索）

このノートブックでは、**Cortex Search** を使って PDF 目論見書・ニュース・商品説明書・アナリストレポートを意味的に検索します。

### このパートで体験できること

1. **AI_PARSE_DOCUMENT**: PDF 目論見書をテキストに変換してチャンク分割
2. **Cortex Search Service の作成**: 4種類のデータにインデックスを作成
3. **キーワード検索 vs セマンティック検索**: 従来の検索との圧倒的な違いを体験
4. **フィルター検索**: 感情・銘柄コードなどで絞り込み
5. **高度な検索機能**: Numeric Boosts（重要度スコア）・Time Decays（時間減衰）

### 🚀 体験ポイント

> **「証券担保ローン」と検索しなくても、**
> **「株を売らずにお金を借りたい」で同じ商品説明書がヒットする！**
>
> 顧客からの相談内容をそのままクエリにできるため、
> 担当者が専門用語を知らなくても適切な情報を引き出せます。

### 前提条件
- `setup.sql` 実行済み（ニュース・ローン書類・アナリストレポート・PDF ステージのデータ投入済み）
- `part1_security.ipynb` 実行済み（`COMPUTE_WH` は `setup.sql` で作成済み）

> ⏱️ **このパートの目安時間: 60分**

In [ ]:
%%sql -r result_env
USE DATABASE SNOWFINANCE_DB;
USE SCHEMA DEMO_SCHEMA;
USE WAREHOUSE COMPUTE_WH;
SELECT CURRENT_DATABASE() AS DB, CURRENT_SCHEMA() AS SCHEMA, CURRENT_WAREHOUSE() AS WH;

## 1. 対象データの確認

Cortex Search Service を作成する前に、**インデックス対象のデータ**を確認します。
データの構造を理解することで、どの列を検索対象にするかを設計できます。

> ⏱️ **目安: 5分**

### 1-0. ファンド目論見書 PDF ステージの確認

`setup.sql` で GitHub リポジトリから **PROSPECTUS_STAGE** にコピー済みの目論見書 PDF を確認します。

> ⏱️ **目安: 1分**

In [ ]:
-- setup.sql でコピー済みの目論見書PDFを確認
LIST @SNOWFINANCE_DB.DEMO_SCHEMA.PROSPECTUS_STAGE;

In [ ]:
%%sql -r result_count
-- 各テーブルのレコード数を確認
SELECT 'NEWS_ARTICLES' AS "テーブル名", COUNT(*) AS "レコード数",
       '50件のマーケットニュース' AS "説明" FROM NEWS_ARTICLES
UNION ALL
SELECT 'LOAN_PRODUCT_DOCS', COUNT(*), 'ローン商品説明書（複数セクションに分割済み）' FROM LOAN_PRODUCT_DOCS
UNION ALL
SELECT 'ANALYST_REPORTS', COUNT(*), 'アナリストレポート（30社分）' FROM ANALYST_REPORTS
ORDER BY "テーブル名";

## 2. AI_PARSE_DOCUMENT で PDF を解析

**AI_PARSE_DOCUMENT** は、PDF などのドキュメントを読み取り、
**Markdown 形式のテキスト**に変換します。

### 変換のイメージ

```
PDF（目論見書）
 ┌──────────────────────┐
 │ 1. ファンドの特色      │       AI_PARSE_DOCUMENT
 │   ・高配当株に投資     │  ─────────────────────→  # 1. ファンドの特色
 │ 2. 投資リスク          │                           ・高配当株に投資
 │   ・市場リスク         │                           # 2. 投資リスク
 │   ・為替リスク         │                           ## 市場リスク ...
 └──────────────────────┘
```

見出し・表・リストなどの構造が Markdown に変換されるため、
後続の **チャンク分割** がセクション単位で正確に行えます。

> ⏱️ **目安: 5〜10分**（PDF 1枚あたり数秒、ファイル数 × 並列処理）

In [ ]:
-- まず1ファイルだけ試してみる（動作確認）
SELECT
    RELATIVE_PATH                    AS "ファイル名",
    SIZE                             AS "ファイルサイズ",
    LEFT(
        PARSE_JSON(
            AI_PARSE_DOCUMENT(
                TO_FILE('@SNOWFINANCE_DB.DEMO_SCHEMA.PROSPECTUS_STAGE', RELATIVE_PATH),
                {'mode': 'LAYOUT'}
            )::VARCHAR
        )['content']::VARCHAR,
        500
    ) AS "解析結果（先頭500字）"
FROM DIRECTORY(@SNOWFINANCE_DB.DEMO_SCHEMA.PROSPECTUS_STAGE)
WHERE RELATIVE_PATH LIKE '%.pdf'
LIMIT 1;

In [ ]:
-- 全PDFを一括解析してテーブルに保存 ()
CREATE OR REPLACE TABLE RAW_FUND_DOCS AS
SELECT
    RELATIVE_PATH                AS FILE_NAME,
    SIZE                         AS FILE_SIZE_BYTES,
    CURRENT_TIMESTAMP()          AS PARSED_AT,
    PARSE_JSON(
        AI_PARSE_DOCUMENT(
            TO_FILE('@SNOWFINANCE_DB.DEMO_SCHEMA.PROSPECTUS_STAGE', RELATIVE_PATH),
            {'mode': 'LAYOUT'}
        )::VARCHAR
    )['content']::VARCHAR AS PARSED_MARKDOWN
FROM DIRECTORY(@SNOWFINANCE_DB.DEMO_SCHEMA.PROSPECTUS_STAGE)
WHERE RELATIVE_PATH LIKE '%.pdf';

SELECT
    FILE_NAME       AS "ファイル名",
    FILE_SIZE_BYTES AS "サイズ",
    LENGTH(PARSED_MARKDOWN) AS "解析後文字数",
    PARSED_AT       AS "解析日時"
FROM RAW_FUND_DOCS
ORDER BY FILE_NAME;

### 2-1. チャンク分割

AI_PARSE_DOCUMENT で得た Markdown テキストを、
**SPLIT_TEXT_MARKDOWN_HEADER** でセクション（見出し）単位に分割します。

### なぜチャンク分割が必要か？

Cortex Search は**テキスト単位でインデックスを作成**します。
目論見書全体（数万字）を 1 チャンクにすると検索精度が下がります。
→ 「2. 投資リスク」「3. 費用」などのセクション単位に分割することで、
  質問に対して**最も関連するセクションだけ**を返せるようになります。

| 分割前 | 分割後 |
|---|---|
| 1ファイル = 1チャンク（数万字） | 1ファイル = 10〜30チャンク（数百字ずつ） |
| 検索精度が低い | セクション単位で精度高く検索 |

> ⏱️ **目安: 2分**

In [ ]:
-- SPLIT_TEXT_RECURSIVE_CHARACTER で均一サイズにチャンク分割
CREATE OR REPLACE TABLE FUND_DOC_CHUNKS AS
SELECT
    r.FILE_NAME,
    CASE r.FILE_NAME
        WHEN 'nomura_index_fund_topix_prospectus.pdf'    THEN '野村インデックスファンド・TOPIX'
        WHEN 'nomura_global_reit_monthly_prospectus.pdf' THEN '野村世界不動産投信(毎月分配型)'
        WHEN 'nomura_wrap_fund_standard_prospectus.pdf'  THEN 'のむラップ・ファンド(普通型)'
        ELSE REPLACE(r.FILE_NAME, '.pdf', '')
    END AS FUND_NAME,
    c.value::VARCHAR               AS CHUNK_CONTENT,
    LENGTH(c.value::VARCHAR)       AS CHUNK_LENGTH,
    c.index                        AS CHUNK_INDEX
FROM RAW_FUND_DOCS r,
LATERAL FLATTEN(
    SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
        r.PARSED_MARKDOWN,
        'markdown',
        1000,
        100
    )
) c
WHERE LENGTH(c.value::VARCHAR) > 50;

SELECT
    FUND_NAME         AS "ファンド名",
    COUNT(*)          AS "チャンク数",
    ROUND(AVG(CHUNK_LENGTH)) AS "平均文字数",
    MIN(CHUNK_LENGTH) AS "最小文字数",
    MAX(CHUNK_LENGTH) AS "最大文字数"
FROM FUND_DOC_CHUNKS
GROUP BY FUND_NAME
ORDER BY FUND_NAME;

In [ ]:
-- チャンクの中身を確認（各ファンドの最初の3チャンク）
SELECT
    FUND_NAME AS "ファンド名",
    CHUNK_INDEX AS "順番",
    CHUNK_LENGTH AS "文字数",
    LEFT(CHUNK_CONTENT, 200) AS "内容（先頭200字）"
FROM FUND_DOC_CHUNKS
WHERE CHUNK_INDEX <= 2
ORDER BY FUND_NAME, CHUNK_INDEX;

## 3. Cortex Search Service の作成

各データソースに対して Cortex Search Service を作成します。

### Cortex Search Service の仕組み

```
CREATE CORTEX SEARCH SERVICE サービス名
ON 検索対象列（テキスト）       ← ここをベクトル化してインデックス化
ATTRIBUTES 絞り込み可能な列    ← フィルタリングに使える列
WAREHOUSE = ウェアハウス名     ← インデックス構築に使用
TARGET_LAG = '更新間隔'
AS ( SELECT ... FROM テーブル );
```

> **Note**: インデックス構築には数分かかります。完了を待たずに次の手順に進めます。
> （`SHOW CORTEX SEARCH SERVICES` で状態を確認できます）

> ⏱️ **目安: 10分**（コマンド実行は即時、インデックス構築は並行して進む）

### 3-1. ニュース記事検索サービス

マーケットニュース50件を検索対象とします。
`IMPORTANCE`（重要度：高/中/低）と `PUBLISH_DATETIME`（発行日時）を含めて、
後の高度な検索機能（Time Decays）で使用します。

In [ ]:
%%sql -r result_news_check
-- NEWS_ARTICLES のデータ確認（CONTENT列がベクトル埋め込み対象）
SELECT
    NEWS_ID,
    PUBLISH_DATE            AS "発行日",
    CATEGORY                AS "カテゴリ",
    LEFT(TITLE, 60)         AS "タイトル",
    SENTIMENT               AS "感情",
    IMPORTANCE              AS "重要度",
    RELATED_SECURITIES      AS "関連銘柄",
    LEFT(CONTENT, 200)      AS "埋め込み対象（CONTENT先頭200字）"
FROM NEWS_ARTICLES
ORDER BY PUBLISH_DATE DESC
LIMIT 5;

In [ ]:
%%sql -r result_cs1
-- Cortex Search Service 1: マーケットニュース
CREATE OR REPLACE CORTEX SEARCH SERVICE NEWS_SEARCH_SERVICE
ON CONTENT
ATTRIBUTES CATEGORY, TITLE, RELATED_SECURITIES, PUBLISH_DATE, SENTIMENT
WAREHOUSE = COMPUTE_WH
TARGET_LAG = '1 day'
AS (
    SELECT
        NEWS_ID,
        PUBLISH_DATE,
        PUBLISH_DATETIME,
        CATEGORY,
        TITLE,
        CONTENT,
        SUMMARY,
        RELATED_SECURITIES,
        SENTIMENT,
        IMPORTANCE
    FROM NEWS_ARTICLES
);
SELECT 'NEWS_SEARCH_SERVICE: 作成コマンド実行完了（インデックス構築中）' AS STATUS;

### 3-2. ローン商品説明書検索サービス

証券担保ローン・教育資金贈与信託などの商品説明書を検索対象とします。

> **💡 これが今回の核心部分です！**
> このサービスを使うと、「証券担保ローン」という専門用語を知らなくても
> 「株を売らずに資金調達したい」というキーワードで商品説明書を発見できます。

In [ ]:
%%sql -r result_loan_check
-- LOAN_PRODUCT_DOCS のデータ確認（CONTENT列がベクトル埋め込み対象）
SELECT
    DOC_ID,
    TITLE                   AS "ドキュメント名",
    SECTION                 AS "セクション",
    LEFT(CONTENT, 200)      AS "埋め込み対象（CONTENT先頭200字）"
FROM LOAN_PRODUCT_DOCS
LIMIT 5;

In [ ]:
%%sql -r result_cs2
-- Cortex Search Service 2: ローン商品説明書
CREATE OR REPLACE CORTEX SEARCH SERVICE LOAN_DOCS_SEARCH_SERVICE
ON CONTENT
ATTRIBUTES PRODUCT_ID, DOC_TYPE, SECTION, TITLE
WAREHOUSE = COMPUTE_WH
TARGET_LAG = '1 day'
AS (
    SELECT
        DOC_ID,
        PRODUCT_ID,
        DOC_TYPE,
        SECTION,
        TITLE,
        CONTENT
    FROM LOAN_PRODUCT_DOCS
);
SELECT 'LOAN_DOCS_SEARCH_SERVICE: 作成コマンド実行完了（インデックス構築中）' AS STATUS;

### 3-3. アナリストレポート検索サービス

アナリストレポートの投資論拠を検索対象とします。
銘柄コード・投資判断・アナリスト名でフィルタリングが可能です。

In [ ]:
%%sql -r result_analyst_check
-- ANALYST_REPORTS のデータ確認（CONTENT列がベクトル埋め込み対象）
SELECT
    REPORT_ID,
    PUBLISH_DATE            AS "発行日",
    SECURITY_CODE           AS "銘柄コード",
    SECURITY_NAME           AS "銘柄名",
    ANALYST_NAME            AS "アナリスト名",
    RATING                  AS "投資判断",
    TARGET_PRICE            AS "目標株価",
    LEFT(CONTENT, 200)      AS "埋め込み対象（CONTENT先頭200字）"
FROM ANALYST_REPORTS
ORDER BY PUBLISH_DATE DESC
LIMIT 5;

In [ ]:
%%sql -r result_cs3
-- Cortex Search Service 3: アナリストレポート
CREATE OR REPLACE CORTEX SEARCH SERVICE ANALYST_REPORT_SEARCH_SERVICE
ON CONTENT
ATTRIBUTES SECURITY_CODE, SECURITY_NAME, RATING, PUBLISH_DATE, ANALYST_NAME
WAREHOUSE = COMPUTE_WH
TARGET_LAG = '1 day'
AS (
    SELECT
        REPORT_ID,
        PUBLISH_DATE,
        SECURITY_CODE,
        SECURITY_NAME,
        ANALYST_NAME,
        RATING,
        TARGET_PRICE,
        REPORT_TITLE,
        EXECUTIVE_SUMMARY,
        INVESTMENT_THESIS AS CONTENT,
        KEY_RISKS
    FROM ANALYST_REPORTS
);
SELECT 'ANALYST_REPORT_SEARCH_SERVICE: 作成コマンド実行完了（インデックス構築中）' AS STATUS;

### 3-4. ファンド目論見書検索サービス

チャンクテーブルを元に、**FUND_DOCS_SEARCH_SERVICE** を作成します。

`FUND_NAME` と `SECTION_HEADER` を `ATTRIBUTES`（フィルタ可能な属性）として設定することで、
特定のファンドだけを絞り込んだ検索ができます。

> ⏱️ **目安: 3分**（インデックス構築は並行処理）

In [ ]:
%%sql -r result_fund_preview
-- FUND_DOC_CHUNKS のデータ確認（CHUNK_CONTENT列がベクトル埋め込み対象）
SELECT
    FUND_NAME               AS "ファンド名",
    CHUNK_INDEX             AS "チャンク番号",
    LEFT(CHUNK_CONTENT, 200) AS "埋め込み対象（CHUNK_CONTENT先頭200字）"
FROM FUND_DOC_CHUNKS
LIMIT 5;

In [ ]:
-- Cortex Search Service: ファンド目論見書
CREATE OR REPLACE CORTEX SEARCH SERVICE FUND_DOCS_SEARCH_SERVICE
ON CHUNK_CONTENT
ATTRIBUTES FUND_NAME
WAREHOUSE = COMPUTE_WH
TARGET_LAG = '1 day'
AS (
    SELECT
        FILE_NAME,
        FUND_NAME,
        CHUNK_INDEX,
        CHUNK_CONTENT
    FROM FUND_DOC_CHUNKS
);

In [ ]:
%%sql -r result_show_cs
-- 作成した Cortex Search Service の状態確認
-- READY になるまでインデックス構築中（数分かかる場合あり）
SHOW CORTEX SEARCH SERVICES IN SCHEMA DEMO_SCHEMA;

## 4. キーワード検索 vs セマンティック検索

### 従来のキーワード検索の限界

SQL の `LIKE` 句による従来のキーワード検索は、**文字の完全一致** しか検索できません。
顧客が「株を担保にお金を借りたい」と言っても、「証券担保ローン」というキーワードがないと見つかりません。

### Cortex Search の革新

**意味（セマンティクス）を理解**した検索ができます。
「株を売らずに資金調達」→「証券担保ローン」を発見！

> ⏱️ **目安: 10分**

> **💡 比較の準備**: 同じ質問をキーワード検索とセマンティック検索で試して、
> 結果の違いを確認してください！

In [ ]:
%%sql -r result_keyword_search
-- 【従来のキーワード検索（LIKE句）】
-- 「株を売らずに資金を調達したい」でLIKE検索 → ヒットしない！
SELECT
    DOC_ID,
    TITLE,
    SECTION,
    LEFT(CONTENT, 200) AS "本文"
FROM LOAN_PRODUCT_DOCS
WHERE CONTENT LIKE '%株を売らずに%'
   OR CONTENT LIKE '%資金を調達%'
   OR TITLE LIKE '%株を売らずに%';
-- 結果: 0件（専門用語「証券担保ローン」がクエリにないため）

In [ ]:
%%sql -r result_semantic_search
-- 【Cortex Search（セマンティック検索）】
-- まったく同じ質問でも、意味を理解してヒットする！
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度スコア",
    f.value:TITLE::STRING                  AS "タイトル",
    f.value:SECTION::STRING                AS "セクション",
    LEFT(f.value:CONTENT::STRING, 300)     AS "本文（先頭300字）"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.LOAN_DOCS_SEARCH_SERVICE',
        '{
            "query": "株を売らずに資金を調達したい",
            "columns": ["DOC_ID", "TITLE", "SECTION", "CONTENT"],
            "limit": 3
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;
-- 結果: 「証券担保ローン」の説明書が上位にヒット！

## 5. 基本的なセマンティック検索

### 検索結果の読み方

| 列名 | 意味 |
|---|---|
| `類似度スコア` | 0〜1の類似度（1に近いほど関連度が高い） |
| `TITLE` | ドキュメントのタイトル |
| `SECTION` | ドキュメントのセクション名 |
| `CONTENT` | 検索対象のテキスト（一部） |

> ⏱️ **目安: 10分**

### 5-1. ローン商品説明書の意味検索

In [ ]:
%%sql -r result_search_loan1
-- 「株を売らずに資金を調達したい」- 証券担保ローンを発見
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度スコア",
    f.value:DOC_ID::STRING                 AS "ドキュメントID",
    f.value:TITLE::STRING                  AS "タイトル",
    f.value:SECTION::STRING                AS "セクション",
    LEFT(f.value:CONTENT::STRING, 200)     AS "本文（先頭200字）"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.LOAN_DOCS_SEARCH_SERVICE',
        '{
            "query": "株を売らずに資金を調達したい",
            "columns": ["DOC_ID", "TITLE", "SECTION", "CONTENT"],
            "limit": 5
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

In [ ]:
%%sql -r result_search_loan2
-- 「孫への教育費用を節税しながら渡す方法」
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度スコア",
    f.value:TITLE::STRING                  AS "タイトル",
    f.value:SECTION::STRING                AS "セクション",
    LEFT(f.value:CONTENT::STRING, 250)     AS "本文（先頭250字）"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.LOAN_DOCS_SEARCH_SERVICE',
        '{
            "query": "孫への教育費用を節税しながら渡す方法",
            "columns": ["TITLE", "SECTION", "CONTENT"],
            "limit": 3
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

### 5-2. ニュース記事の検索

In [ ]:
%%sql -r result_search_news1
-- 「日本銀行の金融政策と株式市場への影響」
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度スコア",
    f.value:PUBLISH_DATE::STRING           AS "発行日",
    f.value:TITLE::STRING                  AS "タイトル",
    f.value:SENTIMENT::STRING              AS "センチメント",
    LEFT(f.value:CONTENT::STRING, 200)     AS "本文（先頭200字）"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.NEWS_SEARCH_SERVICE',
        '{
            "query": "日本銀行の金融政策と株式市場への影響",
            "columns": ["PUBLISH_DATE", "TITLE", "CONTENT", "SENTIMENT"],
            "limit": 5
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

### 5-3. アナリストレポートの検索

In [ ]:
%%sql -r result_search_analyst1
-- 「EV シフトとトヨタの競争優位性」
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度スコア",
    f.value:SECURITY_NAME::STRING          AS "銘柄名",
    f.value:ANALYST_NAME::STRING           AS "アナリスト",
    f.value:RATING::STRING                 AS "投資判断",
    f.value:TARGET_PRICE::STRING           AS "目標株価",
    LEFT(f.value:CONTENT::STRING, 250)     AS "投資根拠（先頭250字）"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.ANALYST_REPORT_SEARCH_SERVICE',
        '{
            "query": "EV シフトとトヨタの競争優位性",
            "columns": ["SECURITY_NAME", "ANALYST_NAME", "RATING", "TARGET_PRICE", "CONTENT"],
            "limit": 3
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

### 5-4. ファンド目論見書の検索デモ

目論見書の検索でも、Part 3 と同様に**意味を理解した検索**が威力を発揮します。

> **例**: 「株価が下がったときに損をするリスク」と検索
> → キーワード検索: 「市場リスク」「価格変動リスク」という見出しがなければヒットしない
> → Cortex Search: 意味を理解して「価格変動リスク」「市場リスク」のセクションを発見！

> ⏱️ **目安: 10分**

In [ ]:
-- 検索1: 「投資リスクが低い安定的なファンドを探したい」
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度スコア",
    f.value:FUND_NAME::STRING                              AS "ファンド名",
    LEFT(f.value:CHUNK_CONTENT::STRING, 300)               AS "内容（先頭300字）"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.FUND_DOCS_SEARCH_SERVICE',
        '{
            "query": "投資リスクが低い安定的な運用ができるファンドの特徴",
            "columns": ["FUND_NAME", "CHUNK_CONTENT"],
            "limit": 5
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

In [ ]:
-- 検索2: 「信託報酬（手数料）の比較」
-- 複数ファンドの費用セクションを横断検索
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度スコア",
    f.value:FUND_NAME::STRING                              AS "ファンド名",
    LEFT(f.value:CHUNK_CONTENT::STRING, 300)               AS "費用の説明"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.FUND_DOCS_SEARCH_SERVICE',
        '{
            "query": "信託報酬 手数料 費用 コスト",
            "columns": ["FUND_NAME", "CHUNK_CONTENT"],
            "limit": 6
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f
ORDER BY 2, 1 DESC;

In [ ]:
-- 検索3: 「為替変動リスクについて知りたい」
-- 特定のリスク要因に関する説明を目論見書から抽出
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度スコア",
    f.value:FUND_NAME::STRING                              AS "ファンド名",
    LEFT(f.value:CHUNK_CONTENT::STRING, 400)               AS "説明内容"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.FUND_DOCS_SEARCH_SERVICE',
        '{
            "query": "為替変動リスク 外貨建て資産 円高 円安の影響",
            "columns": ["FUND_NAME", "CHUNK_CONTENT"],
            "limit": 4
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

## 6. フィルター付き検索

`ATTRIBUTES` で指定した列を使って、**検索結果を絞り込む**ことができます。

**フィルターの書式:**
```json
"filter": {"@eq": {"列名": "値"}}       // 完全一致
"filter": {"@contains": {"列名": "値"}} // 部分一致（文字列）
```

> ⏱️ **目安: 5分**

In [ ]:
%%sql -r result_filter_positive
-- ポジティブなニュースのみ検索（感情フィルター）
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度スコア",
    f.value:PUBLISH_DATE::STRING           AS "発行日",
    f.value:TITLE::STRING                  AS "タイトル",
    f.value:SENTIMENT::STRING              AS "センチメント"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.NEWS_SEARCH_SERVICE',
        '{
            "query": "トヨタ自動車の業績と今後の見通し",
            "columns": ["PUBLISH_DATE", "TITLE", "SENTIMENT", "CONTENT"],
            "filter": {"@eq": {"SENTIMENT": "ポジティブ"}},
            "limit": 5
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

## 7. 高度な検索機能

Cortex Search には、よりインテリジェントな検索を実現するための高度な機能があります。

| 機能 | 説明 | 証券営業での活用例 |
|---|---|---|
| **Numeric Boosts** | 数値列の値でスコアをブースト | 重要度の高いニュースを上位に |
| **Time Decays** | 時間が新しいほど高スコア | 最新のニュースを優先表示 |
| **組み合わせ** | フィルター + ブースト + 減衰を組み合わせ | 最新かつ重要なポジティブニュース |

> ⏱️ **目安: 10分**

### 7-1. フィルター + Time Decay の組み合わせ

**フィルター**（条件絞り込み）と **Time Decay**（最新記事の優先表示）を組み合わせることで、より実用的な検索が実現できます。

**例**: ポジティブなニュースの中から、最新記事を優先して検索

```json
"filter":      {"@eq": {"SENTIMENT": "ポジティブ"}},    // 条件絞り込み
"time_decays": [{"col": "PUBLISH_DATETIME", ...}]       // 最新優先
```

In [ ]:
%%sql -r result_combined_search
-- 組み合わせ検索: フィルター + Time Decay
-- 「ポジティブなニュース」に絞り込み、最新記事を優先表示
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "スコア",
    f.value:PUBLISH_DATE::STRING            AS "発行日",
    f.value:IMPORTANCE::STRING              AS "重要度",
    f.value:SENTIMENT::STRING               AS "センチメント",
    f.value:TITLE::STRING                   AS "タイトル"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.NEWS_SEARCH_SERVICE',
        '{
            "query": "株式投資の好機 企業業績改善",
            "columns": ["TITLE", "CONTENT", "PUBLISH_DATE", "PUBLISH_DATETIME", "IMPORTANCE", "SENTIMENT"],
            "filter": {"@eq": {"SENTIMENT": "ポジティブ"}},
            "time_decays": [{"col": "PUBLISH_DATETIME", "target_time": "2026-04-01T00:00:00", "limit_hours": 720}],
            "limit": 5
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f
ORDER BY 1 DESC;

## 8. 山田様の相談への総合活用

「トヨタ株を売って孫の留学費用を捻出したい」という相談に対して、
3つの Cortex Search Service から関連情報を一括収集します。

> **💡 これが Cortex Agent（Part 4）への橋渡しになります！**
> Part 4 では、このような複数サービスへの並行検索を
> AIエージェントが自動で行ってくれます。

> ⏱️ **目安: 5分**

In [ ]:
%%sql -r result_yamada_comprehensive
-- 山田様（C001）の相談に関連する情報を全サービスから収集
SELECT '【関連ニュース】' AS "種別",
    f.value:PUBLISH_DATE::STRING AS "日付",
    f.value:TITLE::STRING AS "タイトル",
    f.value:SENTIMENT::STRING AS "補足",
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "スコア"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.NEWS_SEARCH_SERVICE',
        '{"query":"トヨタ株 売却 税金 留学 教育費","columns":["PUBLISH_DATE","TITLE","SENTIMENT"],"limit":3}'
    ) AS r
), LATERAL FLATTEN(input => PARSE_JSON(r):results) AS f

UNION ALL

SELECT '【商品説明書】',
    NULL,
    f.value:TITLE::STRING,
    f.value:SECTION::STRING,
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3)
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.LOAN_DOCS_SEARCH_SERVICE',
        '{"query":"株を売らずに孫の教育費を捻出したい","columns":["TITLE","SECTION","CONTENT"],"limit":3}'
    ) AS r
), LATERAL FLATTEN(input => PARSE_JSON(r):results) AS f

UNION ALL

SELECT '【アナリストレポート】',
    f.value:PUBLISH_DATE::STRING,
    f.value:SECURITY_NAME::STRING || ' / ' || f.value:ANALYST_NAME::STRING AS TITLE,
    f.value:RATING::STRING,
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3)
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.ANALYST_REPORT_SEARCH_SERVICE',
        '{"query":"トヨタ自動車 株価 今後の見通し","columns":["SECURITY_NAME","ANALYST_NAME","RATING","CONTENT"],"filter":{"@eq":{"SECURITY_CODE":"7203"}},"limit":3}'
    ) AS r
), LATERAL FLATTEN(input => PARSE_JSON(r):results) AS f

ORDER BY 5 DESC;

## まとめ

Part 4 完了！Cortex Search の全機能を体験しました。

### 作成したオブジェクト

| オブジェクト | 種別 | 役割 |
|---|---|---|
| `RAW_FUND_DOCS` | テーブル | AI_PARSE_DOCUMENT の解析結果（Markdown） |
| `FUND_DOC_CHUNKS` | テーブル | セクション単位のチャンク |
| `NEWS_SEARCH_SERVICE` | Cortex Search | マーケットニュース50件 |
| `LOAN_DOCS_SEARCH_SERVICE` | Cortex Search | ローン商品説明書 |
| `ANALYST_REPORT_SEARCH_SERVICE` | Cortex Search | アナリストレポート30件 |
| `FUND_DOCS_SEARCH_SERVICE` | Cortex Search | ファンド目論見書 PDF |

### 体験した機能

| 機能 | 具体的な効果 |
|---|---|
| **AI_PARSE_DOCUMENT** | PDF をそのままSQLで解析・テキスト化 |
| **キーワード検索 vs セマンティック** | 「証券担保ローン」を知らなくても発見できた |
| **フィルター検索** | ポジティブなニュースのみを条件で絞り込み |
| **Time Decays** | 最新ニュースを優先表示 |
| **組み合わせ検索** | フィルター + Time Decay で精度向上 |

**次のステップ:** `part5_cortex_agent.ipynb` でこれらすべてを統合したエージェントを作成してください。